# Contrail segmentation — U-Net + EfficientNet-B4 on GVCCS

**Dataset:** GVCCS · Zenodo 15743988 · CC BY 4.0 · EUROCONTROL MUAC  
**Model:** U-Net + EfficientNet-B4 (ImageNet pretrained)  
**Estimated time:** ~8–10 h on Kaggle T4 x2  

### Before running
1. Accelerator → **GPU T4 x2**
2. (Optional) Upload `contrail_unet_best-4.pt` as a Kaggle Dataset and add it to this notebook via **Add Data** → your dataset. Training will resume from epoch 4 automatically.
3. **Run All**
4. After completion: right panel → **Output** → download `contrail_unet_best.pt`


In [ ]:
# ── 1. GPU check ──────────────────────────────────────────────────────────────
import torch
import torch.nn as nn

N_GPU = torch.cuda.device_count()
print(f'PyTorch: {torch.__version__}  |  GPUs available: {N_GPU}')
for i in range(N_GPU):
    props = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {props.name}  {props.total_memory/1e9:.1f} GB')

if N_GPU == 0:
    raise RuntimeError('No GPU. Accelerator → GPU T4 x2.')

In [ ]:
# ── 2. Install ────────────────────────────────────────────────────────────────
!pip install segmentation-models-pytorch albumentations -q
print('Done.')

In [ ]:
# ── 3. Paths ──────────────────────────────────────────────────────────────────
import os, glob

WORK_DIR  = '/kaggle/working'
DATA_DIR  = f'{WORK_DIR}/GVCCS/train'
SAVE_DIR  = WORK_DIR                   # weights appear in Output tab

# Look for a pre-trained checkpoint uploaded as Kaggle Dataset
# (Add Data → your dataset containing contrail_unet_best-4.pt)
PRETRAIN_CKPT = None
for pattern in ['/kaggle/input/*/*.pt', '/kaggle/input/*/*/*.pt']:
    found = glob.glob(pattern)
    if found:
        PRETRAIN_CKPT = found[0]
        print(f'Found checkpoint: {PRETRAIN_CKPT}')
        break

if PRETRAIN_CKPT is None:
    print('No checkpoint found — training from ImageNet weights (epoch 1)')

# With 2 GPUs batch size doubles → each epoch ~2× faster
BATCH_SIZE = 8 * max(N_GPU, 1)
IMG_SIZE   = 512
EPOCHS     = 30
START_LR   = 1e-4

print(f'Batch size: {BATCH_SIZE}  |  Image size: {IMG_SIZE}  |  Epochs: {EPOCHS}')

In [ ]:
# ── 4. Download GVCCS (~2.1 GB) ───────────────────────────────────────────────
import json

if not os.path.exists(DATA_DIR):
    print('Downloading GVCCS from Zenodo...')
    !wget -q --show-progress \
        'https://zenodo.org/records/15743988/files/GVCCS.zip?download=1' \
        -O {WORK_DIR}/GVCCS.zip
    print('Extracting...')
    !unzip -q {WORK_DIR}/GVCCS.zip -d {WORK_DIR}/
    !rm {WORK_DIR}/GVCCS.zip
    print('Done.')
else:
    print('GVCCS already present.')

with open(f'{DATA_DIR}/annotations.json') as f:
    _meta = json.load(f)
print(f'Images: {len(_meta["images"])}  |  Annotations: {len(_meta["annotations"])}')

In [ ]:
# ── 5. Dataset ────────────────────────────────────────────────────────────────
import json, cv2, numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

class GVCCSDataset(Dataset):
    def __init__(self, data_dir, transform=None, indices=None):
        self.data_dir  = Path(data_dir)
        self.transform = transform
        with open(self.data_dir / 'annotations.json') as f:
            coco = json.load(f)
        self.ann_by_img = {}
        for ann in coco['annotations']:
            self.ann_by_img.setdefault(ann['image_id'], []).append(ann)
        imgs = coco['images']
        self.images = [imgs[i] for i in indices] if indices is not None else imgs

    def __len__(self): return len(self.images)

    def __getitem__(self, idx):
        meta  = self.images[idx]
        image = cv2.cvtColor(
            cv2.imread(str(self.data_dir / 'images' / meta['file_name'])),
            cv2.COLOR_BGR2RGB)
        h, w  = meta.get('height', 1024), meta.get('width', 1024)
        mask  = np.zeros((h, w), dtype=np.uint8)
        for ann in self.ann_by_img.get(meta['id'], []):
            for seg in ann.get('segmentation', []):
                cv2.fillPoly(mask, [np.array(seg, dtype=np.int32).reshape(-1, 2)], 1)
        if self.transform:
            r = self.transform(image=image, mask=mask)
            return r['image'], r['mask'].unsqueeze(0).float()
        return image, mask

print('GVCCSDataset ready.')

In [ ]:
# ── 6. Transforms & DataLoaders ───────────────────────────────────────────────
import albumentations as A
from albumentations.pytorch import ToTensorV2

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

train_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
    A.GaussianBlur(blur_limit=3, p=0.15),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])
val_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

with open(f'{DATA_DIR}/annotations.json') as f:
    _n = len(json.load(f)['images'])

val_size  = int(_n * 0.1)
train_idx = list(range(_n - val_size))
val_idx   = list(range(_n - val_size, _n))

train_loader = DataLoader(
    GVCCSDataset(DATA_DIR, train_tf, train_idx),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(
    GVCCSDataset(DATA_DIR, val_tf, val_idx),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f'Train: {len(train_loader.dataset)} images ({len(train_loader)} batches)')
print(f'Val:   {len(val_loader.dataset)} images ({len(val_loader)} batches)')

In [ ]:
# ── 7. Model ──────────────────────────────────────────────────────────────────
import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name='efficientnet-b4',
    encoder_weights='imagenet',
    in_channels=3,
    classes=1,
    activation=None,
)

# Resume from checkpoint if provided
start_epoch = 1
best_dice   = 0.0
history     = []

if PRETRAIN_CKPT:
    ckpt = torch.load(PRETRAIN_CKPT, map_location='cpu')
    # Support both full checkpoint dict and bare model state_dict
    if isinstance(ckpt, dict) and 'model_state' in ckpt:
        model.load_state_dict(ckpt['model_state'])
        start_epoch = ckpt.get('epoch', 0) + 1
        best_dice   = ckpt.get('best_dice', 0.0)
        history     = ckpt.get('history', [])
        print(f'Full checkpoint: resuming from epoch {start_epoch}, best dice {best_dice:.4f}')
    else:
        model.load_state_dict(ckpt)          # bare state_dict (e.g. contrail_unet_best-4.pt)
        start_epoch = 5                      # known: saved after epoch 4
        print(f'Model weights loaded. Starting from epoch {start_epoch}.')

# Wrap in DataParallel AFTER loading weights
model = model.cuda()
if N_GPU > 1:
    model = nn.DataParallel(model)
    print(f'DataParallel: using {N_GPU} GPUs')

params = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Parameters: {params:.1f}M')

In [ ]:
# ── 8. Loss & optimizer ───────────────────────────────────────────────────────
dice_loss = smp.losses.DiceLoss(mode='binary')
bce_loss  = smp.losses.SoftBCEWithLogitsLoss()
loss_fn   = lambda p, t: dice_loss(p, t) + bce_loss(p, t)

optimizer = torch.optim.AdamW(model.parameters(), lr=START_LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-6)

# Fast-forward scheduler to current epoch so LR is correct after resume
for _ in range(start_epoch - 1):
    scheduler.step()

print(f'LR at epoch {start_epoch}: {scheduler.get_last_lr()[0]:.2e}')

In [ ]:
# ── 9. Training loop ──────────────────────────────────────────────────────────
from tqdm.notebook import tqdm
import time


def dice_score(logits, target, threshold=0.5):
    pred  = (torch.sigmoid(logits) > threshold).float()
    inter = (pred * target).sum()
    union = pred.sum() + target.sum()
    if union < 1:          # both empty = true negative
        return 1.0
    return (2 * inter / (union + 1e-8)).item()


def save_checkpoint(epoch, is_best=False):
    # Unwrap DataParallel before saving so weights load on any device
    raw_model = model.module if hasattr(model, 'module') else model
    ckpt = {
        'epoch':           epoch,
        'model_state':     raw_model.state_dict(),
        'optimizer_state': optimizer.state_dict(),
        'scheduler_state': scheduler.state_dict(),
        'best_dice':       best_dice,
        'history':         history,
    }
    torch.save(ckpt, f'{SAVE_DIR}/checkpoint_last.pt')   # always overwrite
    if is_best:
        torch.save(raw_model.state_dict(),
                   f'{SAVE_DIR}/contrail_unet_best.pt')  # clean weights only


try:
    for epoch in range(start_epoch, EPOCHS + 1):
        t0 = time.time()

        # ── train ──
        model.train()
        tr_loss = tr_dice = 0.0
        for images, masks in tqdm(train_loader,
                                  desc=f'Ep {epoch:02d}/{EPOCHS} train', leave=False):
            images, masks = images.cuda(), masks.cuda()
            optimizer.zero_grad()
            logits = model(images)
            loss   = loss_fn(logits, masks)
            loss.backward()
            optimizer.step()
            tr_loss += loss.item()
            tr_dice += dice_score(logits, masks)

        # ── val ──
        model.eval()
        vl_loss = vl_dice = 0.0
        with torch.no_grad():
            for images, masks in tqdm(val_loader,
                                      desc=f'Ep {epoch:02d}/{EPOCHS} val  ', leave=False):
                images, masks = images.cuda(), masks.cuda()
                logits = model(images)
                vl_loss += loss_fn(logits, masks).item()
                vl_dice += dice_score(logits, masks)

        scheduler.step()

        nb, nvb   = len(train_loader), len(val_loader)
        ep_dice   = vl_dice / nvb
        is_best   = ep_dice > best_dice
        if is_best:
            best_dice = ep_dice

        row = dict(epoch=epoch,
                   tr_loss=tr_loss/nb, tr_dice=tr_dice/nb,
                   vl_loss=vl_loss/nvb, vl_dice=ep_dice)
        history.append(row)

        # Save checkpoint every epoch — safe to interrupt any time
        save_checkpoint(epoch, is_best=is_best)

        marker = '  ← best' if is_best else ''
        print(f'Ep {epoch:02d}/{EPOCHS}  '
              f'tr loss={row["tr_loss"]:.4f} dice={row["tr_dice"]:.4f}  '
              f'val loss={row["vl_loss"]:.4f} dice={ep_dice:.4f}  '
              f'{time.time()-t0:.0f}s{marker}')

except KeyboardInterrupt:
    print('\nInterrupted — saving current state...')
    save_checkpoint(epoch)

print(f'\nDone. Best val Dice: {best_dice:.4f}')
print(f'Weights: {SAVE_DIR}/contrail_unet_best.pt')
print(f'Resume:  {SAVE_DIR}/checkpoint_last.pt')

In [ ]:
# ── 10. Training curves ───────────────────────────────────────────────────────
import matplotlib.pyplot as plt

epochs_x = [r['epoch'] for r in history]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs_x, [r['tr_loss'] for r in history], label='train')
ax1.plot(epochs_x, [r['vl_loss'] for r in history], label='val')
ax1.set_xlabel('Epoch'); ax1.set_title('Loss (Dice + BCE)'); ax1.legend()

ax2.plot(epochs_x, [r['tr_dice'] for r in history], label='train')
ax2.plot(epochs_x, [r['vl_dice'] for r in history], label='val')
ax2.axhline(0.60, color='orange', linestyle='--', label='demo threshold')
ax2.axhline(0.75, color='green',  linestyle='--', label='PoC threshold')
ax2.set_xlabel('Epoch'); ax2.set_title('Dice score'); ax2.legend()

plt.suptitle(f'Best val Dice = {best_dice:.4f}', fontsize=13)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/training_curves.png', dpi=120)
plt.show()

In [ ]:
# ── 11. Visual predictions on 6 val images ────────────────────────────────────
import random

raw_model = model.module if hasattr(model, 'module') else model
raw_model.eval()

with open(f'{DATA_DIR}/annotations.json') as f:
    all_imgs = json.load(f)['images']

samples = random.sample(val_idx, 6)
fig, axes = plt.subplots(6, 3, figsize=(11, 22))

for row, idx in enumerate(samples):
    meta = all_imgs[idx]
    img  = cv2.cvtColor(
        cv2.imread(f'{DATA_DIR}/images/{meta["file_name"]}'),
        cv2.COLOR_BGR2RGB)
    inp  = val_tf(image=img)['image'].unsqueeze(0).cuda()
    with torch.no_grad():
        pred = torch.sigmoid(raw_model(inp))[0, 0].cpu().numpy()

    axes[row, 0].imshow(img);               axes[row, 0].set_title('Input')
    axes[row, 1].imshow(pred, cmap='hot');  axes[row, 1].set_title('Heatmap')
    axes[row, 2].imshow(pred > 0.5, cmap='gray'); axes[row, 2].set_title('Mask (t=0.5)')
    for ax in axes[row]: ax.axis('off')

plt.suptitle(f'Val samples — Best Dice {best_dice:.4f}', fontsize=13)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/sample_predictions.png', dpi=100)
plt.show()

In [ ]:
# ── 12. Output summary ────────────────────────────────────────────────────────
import os
for fname in ['contrail_unet_best.pt', 'checkpoint_last.pt',
              'training_curves.png', 'sample_predictions.png']:
    path = f'{SAVE_DIR}/{fname}'
    size = os.path.getsize(path) / 1e6 if os.path.exists(path) else 0
    status = f'{size:.1f} MB' if size else 'MISSING'
    print(f'  {fname:<35} {status}')

print(f"""
Download from Kaggle Output tab:
  contrail_unet_best.pt  → edge-pi/python/weights/contrail_unet.pt
  checkpoint_last.pt     → edge-pi/data/  (for resuming next session)

Interpretation:
  Dice > 0.60  → usable for demo
  Dice > 0.75  → production-grade PoC

To resume next Kaggle session:
  Upload checkpoint_last.pt as a Kaggle Dataset → Add Data → run all.
  Training continues from where it stopped.
""")